# Notebook 02 — Cascade and Simulation

**Purpose:** Define the ten-parameter cascade of spatial migration feasibility, quantify its uncertainty through an unconditional variance decomposition, and compute conditional Monte Carlo commitment depths that condition on observed per-hour grid states. Produces the headline simulation results cited in NE manuscript §2 and §5.

**Inputs** (from notebook 01):
- `outputs/contracts/stress_correlation_results.json`
- `outputs/contracts/per_hour_destination_availability.parquet`
- `outputs/contracts/workload_parameters.json`

**Outputs** (consumed by notebook 03):
- `outputs/contracts/cascade_parameters.json`
- `outputs/contracts/conditional_mc_results.json`
- `outputs/tables/sensitivity_surface.csv`
- `outputs/tables/sensitivity_tornado.csv`

---

## Notebook Architecture

| Part | Section | Contents |
|---|---|---|
| **0** | Setup | Imports, REPO_ROOT, contract loading, sanity checks |
| **1** | Cascade Framework (§2) | 10-parameter definitions, ranges, static product, commitment depth |
| **2** | Unconditional Variance Decomposition (§2, Methods) | MC over all parameters, η² attribution, scenario summary |
| **3** | Conditional Monte Carlo (§5) | Single facility, empirical fleet, per-GW sweep |
| **4** | Sensitivity Analysis (§5, Methods) | 2D surface, tornado |
| **5** | Contract Exports | JSONs + sensitivity CSVs |

## Headline Results (reconciled parameter set)

| Claim | Value | Cell |
|---|---|---|
| Cascade central product | ~0.0384 | 1-3 |
| Commitment depth (cascade baseline) | ~20.4% | 1-4 |
| Variance shares D2/D4/D5/S2 | 44/28/13/13% (verify) | 2-1 |
| Single-facility MC commitment depth | TBD from regression | 3-2 |
| Empirical fleet MC commitment depth | TBD from regression | 3-3 |
| Per-GW sweep @ 10 GW | TBD from regression | 3-4 |

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-1: IMPORTS, PATH RESOLUTION, CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

# ─── REPO_ROOT resolver ──────────────────────────────────────────────────────
# Kept in sync with notebooks/01_empirical_evidence.ipynb Cell 2-1. Resolves
# the repo root whether the notebook is launched from the repo root (VS Code,
# Jupyter started at repo root) or from notebooks/ (nbconvert default, or
# Jupyter started inside notebooks/).
# ─────────────────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
if (_cwd / "notebooks").exists() and (_cwd / "data").exists():
    REPO_ROOT = _cwd                      # launched from repo root
elif _cwd.name == "notebooks":
    REPO_ROOT = _cwd.parent               # launched from notebooks/
else:
    REPO_ROOT = _cwd                      # fallback

# Guard against silent mis-resolution: if REPO_ROOT doesn't look like the repo,
# fail loudly here rather than further downstream with a confusing path error.
assert (REPO_ROOT / "notebooks").exists() and (REPO_ROOT / "outputs").exists(), (
    f"REPO_ROOT resolved to {REPO_ROOT}, which does not look like the repo root "
    f"(missing notebooks/ or outputs/). Launch Jupyter from the repo root or "
    f"from notebooks/."
)

# ─── Derived paths ───────────────────────────────────────────────────────────
CONTRACTS_DIR = REPO_ROOT / "outputs" / "contracts"
FIGURES_DIR   = REPO_ROOT / "outputs" / "figures"
TABLES_DIR    = REPO_ROOT / "outputs" / "tables"

CONTRACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

print(f"REPO_ROOT:     {REPO_ROOT}")
print(f"CONTRACTS_DIR: {CONTRACTS_DIR}")

REPO_ROOT:     C:\Users\dunla\repos\data-center-flexibility-resource-adequacy
CONTRACTS_DIR: C:\Users\dunla\repos\data-center-flexibility-resource-adequacy\outputs\contracts


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-2: LOAD CONTRACTS FROM NOTEBOOK 01
# ══════════════════════════════════════════════════════════════════════════════

with open(CONTRACTS_DIR / "stress_correlation_results.json") as f:
    stress_results = json.load(f)

per_hour_avail = pd.read_parquet(CONTRACTS_DIR / "per_hour_destination_availability.parquet")

with open(CONTRACTS_DIR / "workload_parameters.json") as f:
    workload = json.load(f)

# S3 flows through from workload contract — single source of truth for S3=0.90
# across the cascade. Notebook 01 writes it; notebook 02 reads it; notebook 03
# consumes it via the cascade_parameters.json export written in Cell 5-1 below.
S3_FROM_CONTRACT = workload["s3_parameterization"]

print(f"stress_results top-level keys: {list(stress_results.keys())}")
print(f"per_hour_avail: {per_hour_avail.shape} ({len(per_hour_avail.columns)} cols)")
print(f"workload: S3 = {S3_FROM_CONTRACT}")

stress_results top-level keys: ['metadata', 'headline', 'empirical_destination_lmps', 'per_zone', 'yearly']
per_hour_avail: (200, 21) (21 cols)
workload: S3 = {'value': 0.9, 'justification': 'P99 drain times across two independent datasets range from 4.5 to 12.1 seconds at 60 tok/s, approximately two orders of magnitude below PJM 10-minute dispatch window. S3 = 0.90 reserves 10% of load for long-running or batched requests.', 'pjm_dispatch_window_seconds': 600}


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-3: CONTRACT SANITY CHECKS
# ══════════════════════════════════════════════════════════════════════════════
# Fail loudly if upstream contracts are malformed, stale, or missing expected
# fields. These checks guard against silent drift between notebook 01 outputs
# and notebook 02 inputs.
# ══════════════════════════════════════════════════════════════════════════════

# ─── Parquet structural checks ───────────────────────────────────────────────
assert per_hour_avail.shape == (200, 21), (
    f"Unexpected parquet shape: {per_hour_avail.shape}, expected (200, 21)"
)
assert "timestamp" in per_hour_avail.columns, "timestamp column missing"
assert "pjm_co_stressed_mw" in per_hour_avail.columns, "pjm_co_stressed_mw column missing"

dest_cols = [c for c in per_hour_avail.columns if c not in ("timestamp", "pjm_co_stressed_mw")]
assert len(dest_cols) == 19, f"Expected 19 destination columns, got {len(dest_cols)}"

# ─── Cell 4-2 fix verification ───────────────────────────────────────────────
# Pre-fix, notebook 01 filtered dest_zones by 'PJM' string, which never matched
# (all dest zones are cross-BA: CAISO/ERCOT/MISO/NYISO), leaving pjm_co_stressed_mw
# all zeros. Post-fix should have ~30 non-zero hours averaging several GW.
n_nonzero = int((per_hour_avail["pjm_co_stressed_mw"] > 0).sum())
assert n_nonzero > 0, "pjm_co_stressed_mw is all zeros — Cell 4-2 fix did not persist"
assert n_nonzero >= 20, f"Only {n_nonzero} non-zero PJM co-stress hours; expected at least 20"

# ─── Workload contract ───────────────────────────────────────────────────────
assert S3_FROM_CONTRACT == 0.90, f"S3 = {S3_FROM_CONTRACT}, expected 0.90"

# ─── Summary ─────────────────────────────────────────────────────────────────
pjm_mean = per_hour_avail.loc[per_hour_avail["pjm_co_stressed_mw"] > 0, "pjm_co_stressed_mw"].mean()
print(f"✓ Contract sanity checks passed")
print(f"  Stress hours:                   {len(per_hour_avail)}")
print(f"  Destination zones (cross-BA):   {len(dest_cols)}")
print(f"  PJM co-stress non-zero hours:   {n_nonzero} / {len(per_hour_avail)}")
print(f"  PJM co-stress mean (non-zero):  {pjm_mean:,.0f} MW")
print(f"  S3 (from workload contract):    {S3_FROM_CONTRACT}")

AssertionError: S3 = {'value': 0.9, 'justification': 'P99 drain times across two independent datasets range from 4.5 to 12.1 seconds at 60 tok/s, approximately two orders of magnitude below PJM 10-minute dispatch window. S3 = 0.90 reserves 10% of load for long-running or batched requests.', 'pjm_dispatch_window_seconds': 600}, expected 0.90